In [16]:
"""
cnn_model_single.py — Entrenamiento de CNN para UNA combinación de ventanas
=============================================================================

Cambia los parámetros en la sección CONFIGURACIÓN y ejecuta.
"""
import os
os.chdir(r'C:\Users\eneko\OneDrive\Escritorio\MIAX\Practica Redes')
print(os.listdir())
import numpy as np
import sys
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

from config import RANDOM_SEED, MODELS_DIR, FIGURES_DIR
from data_pipeline import load_data, get_train_test
from evaluation import compute_mae, save_results, count_parameters
from plotting import plot_training_curves

np.random.seed(RANDOM_SEED)
import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam


# ============================================================================
# ▶▶▶ CONFIGURACIÓN — CAMBIA AQUÍ LOS PARÁMETROS ◀◀◀
# ============================================================================

INPUT_WINDOW = 90          # Ventana de entrada: 5, 10, 30 o 90
OUTPUT_WINDOW = 1         # Ventana de salida: 1, 5, 30 o 90

FILTERS = 32            # Filtros primera capa Conv1D
KERNEL_SIZE = 7           # Tamaño del kernel primera capa Conv1D
PADDING = 'same'          # 'same' o 'valid'

USE_MAXPOOL = True       # True → usa MaxPooling1D después de la primera Conv1D
POOL_SIZE = 4             # MaxPooling pool_size (solo si USE_MAXPOOL = True)

FILTERS_2 = 64            # Filtros segunda capa Conv1D (poner 0 para no usarla)
KERNEL_SIZE_2 = 7         # Tamaño del kernel segunda capa Conv1D

USE_GAP = False            # True → GlobalAveragePooling1D; False → Flatten

DENSE_1 = 64             # Neuronas primera capa densa
DENSE_2 = 32              # Neuronas segunda capa densa (poner 0 para no usarla)
DENSE_3 = 8               # Neuronas tercera capa densa (poner 0 para no usarla)
DROPOUT_RATE = 0.4        # Dropout (poner 0 para desactivar)
LEARNING_RATE = 0.000001
EPOCHS = 200
BATCH_SIZE = 4
PATIENCE = 120
VALIDATION_SPLIT = 0.1

MODEL_NAME = (f"CNN_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING),
    ]

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu', padding='same'))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE > 0:
        layers.append(Dropout(DROPOUT_RATE))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE > 0:
            layers.append(Dropout(DROPOUT_RATE))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE > 0:
            layers.append(Dropout(DROPOUT_RATE))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()

X_train, X_test, y_train, y_test = get_train_test(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=VALIDATION_SPLIT,
    shuffle=True,
    random_state=RANDOM_SEED
)

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()

"""early_stop = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)"""""

history = model.fit(
    X_tr, y_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    #callbacks=[early_stop],
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

os.makedirs(MODELS_DIR, exist_ok=True)
model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

['baselines.py', 'CNN.ipynb', 'config.py', 'data_pipeline.py', 'evaluation.py', 'Modelo por modelo.ipynb', 'models', 'plotting.py', 'portfolio.py', 'results', '__init__.py', '__pycache__']
Cargando datos...


[*********************100%***********************]  23 of 23 completed


Datos cargados: 16197 días, 23 activos
Rango: 1962-01-03 → 2026-05-12
Ventana entrada=90, salida=1
  X_train: (14496, 90, 23) | y_train: (14496, 23)
  X_test:  (1611, 90, 23)  | y_test:  (1611, 23)

  Modelo: CNN_f32_k7_in90_out1_MP
  Parámetros: 112,311


Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_22 (Conv1D)              │ (None, 90, 32)         │         5,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 22, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_23 (Conv1D)              │ (None, 22, 64)         │        14,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_11 (Flatten)            │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 64)             │        90,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 8)              │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 23)             │           207 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,311 (438.71 KB)

 Trainable params: 112,311 (438.71 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0122 - val_loss: 0.0116
Epoch 2/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0117 - val_loss: 0.0116
Epoch 3/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 4/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 5/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 6/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 7/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 8/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 9/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 10/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 11/200
3262/3262 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0116 - val_loss: 0.0116
Epoch 12/200
3262/3

KeyboardInterrupt: 

In [49]:
import sys, types

# Crear módulo 'src' virtual que apunte a los archivos de la carpeta actual
sys.modules['src'] = types.ModuleType('src')
sys.modules['src.data_pipeline'] = __import__('data_pipeline')
sys.modules['src.evaluation'] = __import__('evaluation')

# Ahora sí funciona
from baselines import predict_naive, predict_zero, predict_mean

import pandas as pd
from config import INPUT_WINDOWS, OUTPUT_WINDOWS
from data_pipeline import load_data, get_train_test
from evaluation import compute_mae
from baselines import predict_naive, predict_zero, predict_mean

returns = load_data()

baselines = {'Naive': predict_naive, 'Zero': predict_zero, 'Mean': predict_mean}
rows = []

INPUT_WINDOW = 5          # Ventana de entrada: 5, 10, 30 o 90
OUTPUT_WINDOW = 5         # Ventana de salida: 1, 5, 30 o 90

for iw in INPUT_WINDOWS:
    for ow in OUTPUT_WINDOWS:
        X_train, X_test, y_train, y_test = get_train_test(returns, iw, ow)
        for name, fn in baselines.items():
            rows.append({
                'Modelo': name,
                'Input': iw,
                'Output': ow,
                'MAE Train': round(compute_mae(y_train, fn(X_train)), 6),
                'MAE Test': round(compute_mae(y_test, fn(X_test)), 6),
            })

tabla = pd.DataFrame(rows)
print(tabla.to_string(index=False))

[*********************100%***********************]  23 of 23 completed


Datos cargados: 16195 días, 23 activos
Rango: 1962-01-03 → 2026-05-08
Ventana entrada=5, salida=1
  X_train: (14571, 5, 23) | y_train: (14571, 23)
  X_test:  (1619, 5, 23)  | y_test:  (1619, 23)
Ventana entrada=5, salida=5
  X_train: (14567, 5, 23) | y_train: (14567, 23)
  X_test:  (1619, 5, 23)  | y_test:  (1619, 23)
Ventana entrada=5, salida=30
  X_train: (14544, 5, 23) | y_train: (14544, 23)
  X_test:  (1617, 5, 23)  | y_test:  (1617, 23)
Ventana entrada=5, salida=90
  X_train: (14490, 5, 23) | y_train: (14490, 23)
  X_test:  (1611, 5, 23)  | y_test:  (1611, 23)
Ventana entrada=10, salida=1
  X_train: (14566, 10, 23) | y_train: (14566, 23)
  X_test:  (1619, 10, 23)  | y_test:  (1619, 23)
Ventana entrada=10, salida=5
  X_train: (14562, 10, 23) | y_train: (14562, 23)
  X_test:  (1619, 10, 23)  | y_test:  (1619, 23)
Ventana entrada=10, salida=30
  X_train: (14540, 10, 23) | y_train: (14540, 23)
  X_test:  (1616, 10, 23)  | y_test:  (1616, 23)
Ventana entrada=10, salida=90
  X_train: (1